In [1]:
import geopandas as gpd
from shapely.geometry import box
import pandas as pd
import numpy as np
from itertools import product
# import math

In [2]:
gdf_limites = gpd.read_file('data/limites_municipio_SP.gpkg')
# gdf_limites = gpd.read_file('data/teste_nomenclatura_folhas.gpkg')
gdf_limites.to_crs("EPSG:4326", inplace=True)
minx, miny, maxx, maxy = gdf_limites.total_bounds
bbox_limites = box(minx, miny, maxx, maxy)

In [3]:
print(bbox_limites)

POLYGON ((-46.36499451495302 -24.00826252550911, -46.36499451495302 -23.35670118518963, -46.82637971457387 -23.35670118518963, -46.82637971457387 -24.00826252550911, -46.36499451495302 -24.00826252550911))


In [4]:
escalas = {
    1000000: {
        "latitude": {
            "graus": 4,
            "linhas": 1,
        },
        "longitude": {
            "graus": 6,
            "colunas": 1,
        },
        "nomenclatura": str(range(0,360))
    },
    500000: {
        "latitude": {
            "graus": 2,
            "linhas": 2,
        },
        "longitude": {
            "graus": 3,
            "colunas": 2,
        },
        "nomenclatura": ["VX", "YZ"] 
    },
    250000: {
        "latitude": {
            "graus": 1,
            "linhas": 2,
        },
        "longitude": {
            "graus": 1.5,
            "colunas": 2,
        },
        "nomenclatura": ["AB", "CD"] 
    },
    100000: {
        "latitude": {
            "graus": 0.5,
            "linhas": 2,
        },
        "longitude": {
            "graus": 0.5,
            "colunas": 3,
        },
        "nomenclatura": [["I", "II", "III"], ["IV", "V", "VI"]] 
    },
    50000: {
        "latitude": {
            "graus": 0.25,
            "linhas": 2,
        },
        "longitude": {
            "graus": 0.25,
            "colunas": 2,
        },
        "nomenclatura": ["12", "34"] 
    },
    25000: {
        "latitude": {
            "graus": 0.125,
            "linhas": 2,
        },
        "longitude": {
            "graus": 0.125,
            "colunas": 2,
        },
        "nomenclatura": [["NO", "NE"], ["SO", "SE"]]  
    },
    10000: {
        "latitude": {
            "graus": 0.125/3,
            "linhas": 3,
        },
        "longitude": {
            "graus": 0.125/2,
            "colunas": 2,
        },
        "nomenclatura": ["AB", "CD", "EF"]  
    }, 
    5000: {
        "latitude": {
            "graus": 0.125/3/2,
            "linhas": 2,
        },
        "longitude": {
            "graus": 0.125/2/2,
            "colunas": 2,
        },
        "nomenclatura": [["I", "II"], ["III", "IV"]]  
    }, 
    2000: {
        "latitude": {
            "graus": 0.125/3/2/2,
            "linhas": 2,
        },
        "longitude": {
            "graus": 0.125/2/2/3,
            "colunas": 3,
        },
        "nomenclatura": ["123", "456"]  
    }
}

In [5]:
def nome_da_folha(lat, long, escala) -> str:

    nome = [] 

    for k in sorted(escalas.keys(), reverse=True):

        g_lat = escalas[k]["latitude"]["graus"]
        l_lat = escalas[k]["latitude"]["linhas"]

        g_long = escalas[k]["longitude"]["graus"]
        c_long = escalas[k]["longitude"]["colunas"]

        nomenclatura = escalas[k]["nomenclatura"][::-1] #inverte a ordem para poder seguir com a atribuição correta

        lat = lat % (g_lat * l_lat)
        long = long % (g_long * c_long)

        lat_indice = int((lat % (g_lat * l_lat)) // g_lat)
        long_indice = int((long % (g_long * c_long)) // g_long)

        if k != 1000000:
            # print(lat, long, lat_indice, long_indice, k)
            nome.append(nomenclatura[lat_indice][long_indice])
        if k <= escala:
            break
    return "-".join(nome)

In [6]:
nome_da_folha(-24,-48, 2000)

'Y-C-IV-3-SO-E-III-4'

In [7]:
def cria_folhas(bbox, escala:int) -> gpd.GeoDataFrame:
    minx, miny, maxx, maxy = bbox.bounds
    variacao_x = escalas[escala]["longitude"]["graus"]
    variacao_y = escalas[escala]["latitude"]["graus"]
    x_inicial = (minx // (escalas[escala]["longitude"]["graus"])) * (escalas[escala]["longitude"]["graus"])
    y_inicial = (miny // (escalas[escala]["latitude"]["graus"])) * (escalas[escala]["latitude"]["graus"])
    x0 = np.arange(x_inicial, maxx, variacao_x)
    # x1 = x0 + variacao_x
    y0 = np.arange(y_inicial, maxy, variacao_y)
    # y1 = y0 + variacao_y
    boxes = []
    nomes = []
    for x in x0:
        for y in y0:
            poligono = box(x, y, x + variacao_x, y + variacao_y)
            boxes.append(poligono)
            nomes.append(nome_da_folha(poligono.centroid.y, poligono.centroid.x, escala))
    return gpd.GeoDataFrame({"nome":nomes}, geometry=boxes, crs="EPSG:4326")

    # return x_inicial


In [8]:
cria_folhas(bbox_limites, 2000)

,nome,geometry
0,V-A-III-1-NE-A-II-2,"POLYGON ((-46.823 -24.01, -46.823 -24, -46.833..."
1,Y-C-VI-3-SE-E-IV-5,"POLYGON ((-46.823 -24, -46.823 -23.99, -46.833..."
2,Y-C-VI-3-SE-E-IV-2,"POLYGON ((-46.823 -23.99, -46.823 -23.979, -46..."
3,Y-C-VI-3-SE-E-II-5,"POLYGON ((-46.823 -23.979, -46.823 -23.969, -4..."
4,Y-C-VI-3-SE-E-II-2,"POLYGON ((-46.823 -23.969, -46.823 -23.958, -4..."
...,...,...
2830,Y-D-I-3-SE-A-III-1,"POLYGON ((-46.365 -23.406, -46.365 -23.396, -4..."
2831,Y-D-I-3-SE-A-I-4,"POLYGON ((-46.365 -23.396, -46.365 -23.385, -4..."
2832,Y-D-I-3-SE-A-I-1,"POLYGON ((-46.365 -23.385, -46.365 -23.375, -4..."
2833,Y-D-I-3-NE-E-III-4,"POLYGON ((-46.365 -23.375, -46.365 -23.365, -4..."


In [9]:
cria_folhas(bbox_limites, 2000).to_file('results/folhas_sp.gpkg', driver="GPKG")

In [10]:
gdf_folhas  = gpd.read_file('results/folhas_sp.gpkg')

In [24]:
gdf_folhas[gdf_folhas.intersects(gdf_limites.unary_union)].to_file('results/folhas_sp_cortada.gpkg')

/var/folders/4v/rx_d3gzj4991_pw5_skhl0980000gn/T/ipykernel_14034/2160154946.py:1: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  gdf_folhas[gdf_folhas.intersects(gdf_limites.unary_union)].to_file('results/folhas_sp_cortada.gpkg')


In [21]:
gdf_limites

,primaryindex,sp_areamt,sp_areakmt,sp_codigo,sp_id,sp_sigla,sp_nome,geometry
0,1,31980202.74,32.0,03,2.0,FO,FREGUESIA-BRASILANDIA,"MULTIPOLYGON (((-46.812 -23.91, -46.812 -23.91..."
